# Byte Pair Encoding (BPE) From Scratch

Today's drill is to implement a Byte Pair Encoding (BPE) tokenizer from scratch using a byte-level vocabulary. BPE is a subword 
tokenization algorithm that learns frequently occuring byte sequences and merges them into new vocabulary entries.

The tokenizer will:
- Start with UTF-8 bytes (0-255) as the base vocabulary.
- Learn merge rules from a training corpus.
- Encode text by greedily applying learned merges.
- Decode token IDs back into text.
- Analyze vocabulary efficienct and compression behavior.

### Requirements

1. Byte-Level Vocabulary
* UTF-8 bytes (0-255) form the inital vocabulary.
* Every text sequence can be represented without requiring an unknown token.
* New vocabulary entries begin at ID 256

2. BPE Training Algorithm

    The training loop follows an iterative merge process:

    Current Token IDs -> Count Adjacent Pairs -> Select Most Frequent Pair -> Assign New Token ID -> Merge Occurences -> Record Merge Rule -> Repeat

    Example:

    - [h, e, l, l, o]

    - Most Frequent Pair = (h, e)

    - Assign 256 -> (h, e)

    - Result = [256, l, l, o]

3. Pair Frequency Computation

    For a token sequence:

    - T = [t<sub>1</sub>, t<sub>2</sub>, ..., t<sub>n</sub>]

    Count every adjacent pair:

    - (t<sub>1</sub>, t<sub>i+1</sub>)



4. Greedy Encoding

    When encoding new text:
    1. Convert text into UTF-8 bytes
    2. Apply learned merge rules in the exact order they were discovered
    3. Continue until no applicable merges remain

    This ensures consistency between training and inference. 

5. Decoder Construction

    - IDs 0-255 map directly to bytes
    - IDs greater than 255 recursively expand into the byte sequences that created them
    - The final bytes stream is converted back into text using UTF-8 decoding with errors = "replace"

### Expected Output

```
BYTE PAIR TOKENIZER (v1)

TRAINING STATISTICS
Initial Vocabulary Size: 256
Target Vocabulary Size: 275
Learned Merges: 19

ENCODING DEMONSTRATION

Input text: 
tokenization

Encoded IDs:
[...]

Decoded Text:
tokenization

VOCABULARY EFFICIENCY ANALYSIS

Original Character Count: ...
Encoded Token Count: ...
Compression Ratios: ...

COMMON WORD ANALYSIS
Word: the
Encoded IDs: [...]

RARE WORD ANALYSIS
Word: hallucination
Encoded IDs: [...]

### Imports

In [119]:
from collections import Counter
import numpy as np

### Raw Training Corpus

In [120]:
corpus = """
tokenization is the process of breaking down text into smaller units.
subword tokenization is a sweet spot betweeen word and character levels.
it helps model handle rare words by using common sub-pieces.
"""

target_vocab_size = 275

### Byte Conversion Pipeline

In [121]:
def text_to_bytes(text):
    """Converts text into a list of UTF-8 byte IDs."""

    return list(text.encode("utf-8"))

In [122]:
# Verify function
test_text = "this is a test"
ttb = text_to_bytes(test_text)
print(ttb)

[116, 104, 105, 115, 32, 105, 115, 32, 97, 32, 116, 101, 115, 116]


### Pair Counting Engine

In [123]:
def count_pairs(token_ids):
    """Counts frequencies of adjacent token pairs."""
    pair_counts = Counter()

    for i in range(len(token_ids) - 1):
        pair = (token_ids[i], token_ids[i + 1])
        pair_counts[pair] += 1

    return pair_counts

In [124]:
# Verify Function
cp = count_pairs(ttb)
print(cp)

Counter({(105, 115): 2, (115, 32): 2, (116, 104): 1, (104, 105): 1, (32, 105): 1, (32, 97): 1, (97, 32): 1, (32, 116): 1, (116, 101): 1, (101, 115): 1, (115, 116): 1})


### Merge Engine

In [125]:
def merge_tokens(token_ids, pair, new_id):
    """Relaces all occurrences of a token pair with a new toekn ID."""
    merged = []
    i = 0

    while i < len(token_ids):
        if (i < len(token_ids) - 1 and token_ids[i] == pair[0] and token_ids[i + 1] == pair[1]):
            merged.append(new_id)
            i += 2  # Skip the next token since it's part of the pair
        else:
            merged.append(token_ids[i])
            i += 1

    return merged

In [126]:
# Verify Function
mt = merge_tokens(ttb, (116, 104), 256) # Merging "t" into a new token ID 256
print(mt)

[256, 105, 115, 32, 105, 115, 32, 97, 32, 116, 101, 115, 116]


### BPE Training Engine

In [127]:
def learn_merges(text, target_vocab_size):
    """Learns BPE merge rules until the target vocabulary size is reached."""
    token_ids = text_to_bytes(text)
    merges = {}
    next_id = 256  # Start assigning new token IDs from 256

    while next_id < target_vocab_size:
        # Count pair frequencies
        pair_counts = count_pairs(token_ids) 

        if not pair_counts:
            break # No more pairs to merge

        # Select most frequent pair
        best_pair = max(pair_counts, key=pair_counts.get)

        # Record merge rule
        merges[best_pair] = next_id

        # Apply merge
        token_ids = merge_tokens(token_ids, best_pair, next_id)
        next_id += 1

    return merges

In [128]:
# Verify function
lm = learn_merges(test_text, 266)
print(lm)

{(105, 115): 256, (256, 32): 257, (116, 104): 258, (258, 257): 259, (259, 257): 260, (260, 97): 261, (261, 32): 262, (262, 116): 263, (263, 101): 264, (264, 115): 265}


### Decoder Construction

In [129]:
def build_decoder(merges):
    """ Builds an ID-to-byte mapping from learned merge rules."""
    vocab = {}

    # Base byte vocabulary
    for idx in range(256):
        vocab[idx] = bytes([idx])

    # Learned merge vocabulary
    for pair, token_id in merges.items():
        vocab[token_id] = (vocab[pair[0]] + vocab[pair[1]])

    return vocab

In [130]:
# Verify Function
bd = build_decoder(lm)
print(bd)

{0: b'\x00', 1: b'\x01', 2: b'\x02', 3: b'\x03', 4: b'\x04', 5: b'\x05', 6: b'\x06', 7: b'\x07', 8: b'\x08', 9: b'\t', 10: b'\n', 11: b'\x0b', 12: b'\x0c', 13: b'\r', 14: b'\x0e', 15: b'\x0f', 16: b'\x10', 17: b'\x11', 18: b'\x12', 19: b'\x13', 20: b'\x14', 21: b'\x15', 22: b'\x16', 23: b'\x17', 24: b'\x18', 25: b'\x19', 26: b'\x1a', 27: b'\x1b', 28: b'\x1c', 29: b'\x1d', 30: b'\x1e', 31: b'\x1f', 32: b' ', 33: b'!', 34: b'"', 35: b'#', 36: b'$', 37: b'%', 38: b'&', 39: b"'", 40: b'(', 41: b')', 42: b'*', 43: b'+', 44: b',', 45: b'-', 46: b'.', 47: b'/', 48: b'0', 49: b'1', 50: b'2', 51: b'3', 52: b'4', 53: b'5', 54: b'6', 55: b'7', 56: b'8', 57: b'9', 58: b':', 59: b';', 60: b'<', 61: b'=', 62: b'>', 63: b'?', 64: b'@', 65: b'A', 66: b'B', 67: b'C', 68: b'D', 69: b'E', 70: b'F', 71: b'G', 72: b'H', 73: b'I', 74: b'J', 75: b'K', 76: b'L', 77: b'M', 78: b'N', 79: b'O', 80: b'P', 81: b'Q', 82: b'R', 83: b'S', 84: b'T', 85: b'U', 86: b'V', 87: b'W', 88: b'X', 89: b'Y', 90: b'Z', 91: b'[',

### Encoding Engine

In [131]:
def encode(text, merges):
    """Encodes text using learned NPE merge rules."""
    token_ids = text_to_bytes(text)

    # Apply merges in learned order
    for pair, token_id in merges.items():
        while True:
            updated = merge_tokens(token_ids, pair, token_id)

            if updated == token_ids:
                break

            token_ids = updated
    
    return token_ids

In [132]:
# Verify Function
et = encode(test_text, lm)
print(et)

[265, 116]


### Decoding Engine

In [133]:
def decode(token_ids, vocab):
    """Decodes token IDs back into text."""
    byte_stream = bytearray()

    for token_id in token_ids:
        byte_stream.extend(vocab[token_id])

    return byte_stream.decode("utf-8", errors="replace") # Handle decoding errors gracefully

In [134]:
# Verify Function
dt = decode(et, bd)
print(dt)

this is a test


### Vocabulary Efficiency Analysis

In [135]:
def analyze_compression(text, encoded_ids):
    """Computes compression statistics for encoded text."""
    character_count = len(text)
    token_count = len(encoded_ids)
    compression_ratio = token_count / character_count if character_count > 0 else 0

    return {
        "character_count": character_count, 
        "token_count": token_count,
        "compression_ratio": compression_ratio
    }

In [136]:
# Verify Function
ac = analyze_compression(test_text, et)
print(ac)

{'character_count': 14, 'token_count': 2, 'compression_ratio': 0.14285714285714285}


### Subword Fragmentation Analysis

In [137]:
def analyze_fragmentation(common_word, rare_word, merges):
    """Analyzes how BPE tokenizes common and rare words."""

    common_encoded = encode(common_word, merges)

    rare_encoded = encode(rare_word, merges)

    return {
        "common_word": common_word,
        "common_tokens": common_encoded,
        "common_count": len(common_encoded),
        "rare_word": rare_word,
        "rare_tokens": rare_encoded,
        "rare_count": len(rare_encoded)
    }

In [138]:
# Verify Function
af = analyze_fragmentation("tokenization", "subwordtokenization", lm)
print(af)

{'common_word': 'tokenization', 'common_tokens': [116, 111, 107, 101, 110, 105, 122, 97, 116, 105, 111, 110], 'common_count': 12, 'rare_word': 'subwordtokenization', 'rare_tokens': [115, 117, 98, 119, 111, 114, 100, 116, 111, 107, 101, 110, 105, 122, 97, 116, 105, 111, 110], 'rare_count': 19}


### Tokenization Interpretation Engine

In [139]:
def interpret_fragmentation(fragmentation):
    """Generates a simple interpretation of tokenization behavior."""

    common_count = fragmentation["common_count"]
    rare_count = fragmentation["rare_count"]

    if rare_count > common_count:
        interpretation = (
            "The rare word is fragmented into more subword units "
            "than the common word, demonstrating how BPE handles "
            "previously unseen vocabulary through reusable subword pieces."
        )
    else:
        interpretation = (
            "Both words are represented with a similar number of "
            "subword units under the learned vocabulary."
        )

    return interpretation

In [140]:
# Verify Function
inf = interpret_fragmentation(af)
print(inf)

The rare word is fragmented into more subword units than the common word, demonstrating how BPE handles previously unseen vocabulary through reusable subword pieces.


### Execution and Evalutaion Harness

In [141]:
def evaluate_bpe_pipeline(corpus, target_vocab_size):
    """Trains the tokenizer and reports encoding statistics."""

    print("BYTE PAIR ENCODING TOKENIZER (v1)\n")

    merges = learn_merges(corpus, target_vocab_size)
    vocab = build_decoder(merges)

    print("TRAINING STATISTICS")
    print("Initial Vocabulary Size: 256")
    print(f"Target Vocabulary Size: {target_vocab_size}")
    print(f"Learned Merges: {len(merges)}\n")

    sample_text = "tokenization"
    encoded = encode(sample_text, merges)
    decoded = decode(encoded, vocab)

    print("ENCODING DEMONSTRATION\n")
    print(f"Input Text: {sample_text}")
    print(f"Encoded IDs: {encoded}")
    print(f"Decoded Text: {decoded}\n")

    corpus_encoded = encode(corpus, merges)
    stats = analyze_compression(corpus, corpus_encoded)

    print("VOCABULARY EFFICIENCY ANALYSIS\n")
    print(f"Original Character Count: " f"{stats['character_count']}")
    print(f"Encoded Token Count: " f"{stats['token_count']}")
    print(f"Compression Ratio: " f"{stats['compression_ratio']:.4f}\n")

    common_word = "the"
    rare_word = "hallucination"

    fragmentation = analyze_fragmentation(common_word, rare_word, merges)

    print("COMMON WORD ANALYSIS")
    print(f"Word: {fragmentation['common_word']}")
    print(f"Token Count: {fragmentation['common_count']}")
    print(f"Encoded IDs: {fragmentation['common_tokens']}\n")

    print("RARE WORD ANALYSIS")
    print(f"Word: {fragmentation['rare_word']}")
    print(f"Token Count: {fragmentation['rare_count']}")
    print(f"Encoded IDs: {fragmentation['rare_tokens']}\n")

### Execute BPE Pipeline

In [142]:
evaluate_bpe_pipeline(corpus, target_vocab_size)

BYTE PAIR ENCODING TOKENIZER (v1)

TRAINING STATISTICS
Initial Vocabulary Size: 256
Target Vocabulary Size: 275
Learned Merges: 19

ENCODING DEMONSTRATION

Input Text: tokenization
Encoded IDs: [274, 97, 116, 105, 111, 110]
Decoded Text: tokenization

VOCABULARY EFFICIENCY ANALYSIS

Original Character Count: 205
Encoded Token Count: 148
Compression Ratio: 0.7220

COMMON WORD ANALYSIS
Word: the
Token Count: 3
Encoded IDs: [116, 104, 101]

RARE WORD ANALYSIS
Word: hallucination
Token Count: 12
Encoded IDs: [104, 97, 108, 108, 117, 99, 263, 97, 116, 105, 111, 110]

